# Delta-Gamma Risk Surface

**Multivariable Calculus in Real Derivatives Risk Management**

*This is what multivariable calculus actually looks like in finance.*

Every risk manager on a derivatives desk lives and breathes the geometry of this notebook:  
- The **risk surface** of a portfolio value $V(S_1, S_2)$
- First-order sensitivities (**Delta** = gradient)  
- Second-order sensitivities (**Gamma** = Hessian matrix)  
- Directional derivatives and principal curvature directions  
- 2nd-order Taylor approximations used for lightning-fast P&L attribution  

**Why it matters:**  
In practice, repricing a full book of thousands of derivatives is too slow for real-time risk. Delta-gamma approximation gives instantaneous, accurate-enough P&L surfaces that risk desks monitor every second.


### 1. Import libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import warnings

from plotly.subplots import make_subplots
from scipy.stats import norm

warnings.filterwarnings("ignore")

### 2. Mathematical Foundation

#### 2nd-Order Taylor Expansion in Two Variables

For a portfolio value $V(S_1, S_2)$ around the current point $(S_1^0, S_2^0)$:

$$
\Delta V \approx 
\underbrace{\delta_1 \Delta S_1 + \delta_2 \Delta S_2}_{\text{Delta (linear)}} 
+ 
\frac{1}{2} \underbrace{
\begin{bmatrix} \Delta S_1 & \Delta S_2 \end{bmatrix}
\begin{bmatrix}
\Gamma_{11} & \Gamma_{12} \\
\Gamma_{21} & \Gamma_{22}
\end{bmatrix}
\begin{bmatrix} \Delta S_1 \\ \Delta S_2 \end{bmatrix}
}_{\text{Gamma (quadratic)}}
$$

Where:
- $\delta_i = \frac{\partial V}{\partial S_i}$ → **Delta (gradient)**
- $\Gamma_{ij} = \frac{\partial^2 V}{\partial S_i \partial S_j}$ → **Gamma (Hessian)**
- $\Gamma_{12} = \Gamma_{21}$ → **Symmetry (Clairaut’s theorem)**

**Key Intuitions**
- The **gradient** $\nabla V$ points in the direction of steepest ascent  
- The **Hessian eigenvalues** determine principal curvature directions (max/min risk)  
- The **cross-gamma** $\Gamma_{12}$ captures interaction between the two assets (correlation risk)

### 3. Realistic Portfolio Example: Two Vanilla European Calls

We use a portfolio of two independent European call options (classic desk example).

In [2]:
def black_scholes_call(S, K, T, r, sigma):
  """Analytical Black-Scholes call price and greeks"""
  if T <= 0 or sigma <= 0:
    return max(S - K, 0), 1 if S > K else 0, 0, 0
  
  d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
  d2 = d1 - sigma * np.sqrt(T)
  
  price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
  delta = norm.cdf(d1)
  gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
  # vega and theta omitted for this project
  
  return price, delta, gamma

In [3]:
# Portfolio parameters (realistic desk values)
S1_0, S2_0 = 100.0, 100.0
K1, K2 = 100.0, 100.0
T = 1.0
r = 0.05
sigma1 = 0.25
sigma2 = 0.30

# Current portfolio value and greeks
V0, delta1, gamma11 = black_scholes_call(S1_0, K1, T, r, sigma1)
_, delta2, gamma22 = black_scholes_call(S2_0, K2, T, r, sigma2)

# Cross-gamma = 0 because the options are independent
gamma12 = 0.0

print(f"Current portfolio value V₀ = {V0:.4f}")
print(f"Delta vector   = [{delta1:.4f}, {delta2:.4f}]")
print(f"Gamma matrix   = [[{gamma11:.4f}, {gamma12:.4f}], [{gamma12:.4f}, {gamma22:.4f}]]")

Current portfolio value V₀ = 12.3360
Delta vector   = [0.6274, 0.6243]
Gamma matrix   = [[0.0151, 0.0000], [0.0000, 0.0126]]


### 4. Building the Risk Surface (3D Visualization)

We shock both underlyings on a grid and compute:
- True ΔV (exact repricing)
- Delta approximation
- Delta-Gamma approximation
- Error surface

In [4]:
# Create shock grid (±30% moves)
n_points = 50
ds1_range = np.linspace(-30, 30, n_points)
ds2_range = np.linspace(-30, 30, n_points)
DS1, DS2 = np.meshgrid(ds1_range, ds2_range)

# True portfolio value after shocks
V_true = np.zeros_like(DS1)
for i in range(n_points):
  for j in range(n_points):
    S1_new = S1_0 + DS1[i, j]
    S2_new = S2_0 + DS2[i, j]
    v1, _, _ = black_scholes_call(S1_new, K1, T, r, sigma1)
    v2, _, _ = black_scholes_call(S2_new, K2, T, r, sigma2)
    V_true[i, j] = v1 + v2

delta_V_true = V_true - V0

# Delta (1st-order) approximation
delta_approx = delta1 * DS1 + delta2 * DS2

# Delta-Gamma (2nd-order) approximation
gamma_approx = (0.5 * gamma11 * DS1**2 + gamma12 * DS1 * DS2 + 0.5 * gamma22 * DS2**2)
delta_gamma_approx = delta_approx + gamma_approx
error = delta_V_true - delta_gamma_approx

Visualise the result

In [5]:
fig = make_subplots(
  rows=2, cols=2,
  specs=[[{'type': 'surface'}, {'type': 'surface'}],[{'type': 'surface'}, {'type': 'surface'}]],
  subplot_titles=("True ΔV Surface", "Delta Approximation", "Delta-Gamma Approximation", "Approximation Error")
)

fig.add_trace(go.Surface(x=DS1, y=DS2, z=delta_V_true, colorscale='RdBu_r', showscale=False), row=1, col=1) # True
fig.add_trace(go.Surface(x=DS1, y=DS2, z=delta_approx, colorscale='RdBu_r', showscale=False), row=1, col=2) # Delta
fig.add_trace(go.Surface(x=DS1, y=DS2, z=delta_gamma_approx, colorscale='RdBu_r', showscale=False), row=2, col=1) # Delta-Gamma
fig.add_trace(go.Surface(x=DS1, y=DS2, z=error, colorscale='RdBu_r', showscale=True), row=2, col=2) # Error

fig.update_layout(
  title="Delta-Gamma Risk Surfaces (Portfolio of Two Calls)",
  scene=dict(xaxis_title='ΔS₁', yaxis_title='ΔS₂', zaxis_title='ΔV'),
  height=900, width=1100
)
fig.show()

### 5. Contour Plots & Gradient Visualization

In [6]:
fig = go.Figure()

# True ΔV contours
fig.add_trace(go.Contour(x=ds1_range, y=ds2_range, z=delta_V_true, contours=dict(coloring='heatmap'), name='True ΔV'))

# Gradient arrows (Delta) at center
scale = 15
fig.add_trace(go.Scatter(x=[0], y=[0], mode='markers+text', marker=dict(size=12, color='red'), text=['Current point'], textposition='top center'))

# Arrow in direction of gradient
fig.add_annotation(x=delta1*scale, y=delta2*scale, ax=0, ay=0, xref='x', yref='y', axref='x', ayref='y', arrowhead=2, arrowsize=2, arrowwidth=2, arrowcolor='red', text="∇V (Delta)")
fig.update_layout(title="Contour of True ΔV with Delta Vector", xaxis_title='ΔS₁', yaxis_title='ΔS₂', template='plotly_white', height=600)
fig.show()

### 6. Directional Derivatives & Principal Curvatures

In [7]:
def directional_derivative(delta_vec, gamma_mat, direction):
  """First- and second-order directional derivatives"""
  unit_dir = direction / np.linalg.norm(direction)
  d1 = np.dot(delta_vec, unit_dir)
  d2 = unit_dir.T @ gamma_mat @ unit_dir
  return d1, d2

delta_vec = np.array([delta1, delta2])
gamma_mat = np.array([[gamma11, gamma12], [gamma12, gamma22]])

# Three directions
directions = {
  'Along S1': np.array([1.0, 0.0]),
  'Along S2': np.array([0.0, 1.0]),
  'Equal move': np.array([1.0, 1.0]),
  'Max curvature': None  # computed below
}

print("Directional risk (1st-order / 2nd-order curvature):")
for name, d in directions.items():
  if d is not None:
    d1, d2 = directional_derivative(delta_vec, gamma_mat, d)
    print(f"  {name:12}: ΔV¹ = {d1:6.4f}   Curvature = {d2:6.4f}")

# Eigen-decomposition of Gamma (principal curvatures)
eigvals, eigvecs = np.linalg.eigh(gamma_mat)
print(f"\nPrincipal curvatures (Hessian eigenvalues): {eigvals}")
print(f"Direction of maximum curvature: {eigvecs[:, np.argmax(eigvals)]}")

Directional risk (1st-order / 2nd-order curvature):
  Along S1    : ΔV¹ = 0.6274   Curvature = 0.0151
  Along S2    : ΔV¹ = 0.6243   Curvature = 0.0126
  Equal move  : ΔV¹ = 0.8851   Curvature = 0.0139

Principal curvatures (Hessian eigenvalues): [0.01264776 0.01513679]
Direction of maximum curvature: [1. 0.]


### 7. Delta-Gamma Approximation in Risk Systems

In [8]:
# Simulate a large shock and compare approximations
shock = np.array([15.0, -12.0])   # realistic intraday move

# Exact
S1_shock, S2_shock = S1_0 + shock[0], S2_0 + shock[1]
V_shock, _, _ = black_scholes_call(S1_shock, K1, T, r, sigma1)
V_shock += black_scholes_call(S2_shock, K2, T, r, sigma2)[0]
true_dv = V_shock - V0

# Approximations
delta_approx_shock = np.dot(delta_vec, shock)
gamma_approx_shock = 0.5 * shock.T @ gamma_mat @ shock
dg_approx_shock = delta_approx_shock + gamma_approx_shock

print(f"True ΔV          : {true_dv:8.4f}")
print(f"Delta approx     : {delta_approx_shock:8.4f} (error {abs(true_dv-delta_approx_shock):.4f})")
print(f"Delta-Gamma approx: {dg_approx_shock:8.4f} (error {abs(true_dv-dg_approx_shock):.4f})")

True ΔV          :  18.6205
Delta approx     :   1.9201 (error 16.7004)
Delta-Gamma approx:   4.5336 (error 14.0869)


### 8. Extension: Synthetic Cross-Gamma (Rainbow / Basket Exposure)

In practice many products have non-zero cross-gamma (e.g., basket options, spread options).

In [9]:
# Synthetic portfolio with artificial cross exposure
def synthetic_portfolio(S1, S2):
  v1, _, _ = black_scholes_call(S1, K1, T, r, sigma1)
  v2, _, _ = black_scholes_call(S2, K2, T, r, sigma2)
  cross_term = 0.0008 * (S1 - S1_0) * (S2 - S2_0)   # mimics correlation risk
  return v1 + v2 + cross_term

# Recompute full surface with cross term
V_true_cross = np.zeros_like(DS1)
for i in range(n_points):
  for j in range(n_points):
    V_true_cross[i, j] = synthetic_portfolio(S1_0 + DS1[i,j], S2_0 + DS2[i,j])

delta_V_true_cross = V_true_cross - synthetic_portfolio(S1_0, S2_0)

# Numerical Hessian at center (finite differences)
h = 0.01
gamma11_num = (synthetic_portfolio(S1_0+h, S2_0) - 2*synthetic_portfolio(S1_0, S2_0) + synthetic_portfolio(S1_0-h, S2_0)) / h**2
gamma22_num = (synthetic_portfolio(S1_0, S2_0+h) - 2*synthetic_portfolio(S1_0, S2_0) + synthetic_portfolio(S1_0, S2_0-h)) / h**2
gamma12_num = (synthetic_portfolio(S1_0+h, S2_0+h) - synthetic_portfolio(S1_0+h, S2_0-h) - synthetic_portfolio(S1_0-h, S2_0+h) + synthetic_portfolio(S1_0-h, S2_0-h)) / (4 * h**2)

print(f"Numerical cross-gamma Γ₁₂ = {gamma12_num:.6f} (non-zero!)")

Numerical cross-gamma Γ₁₂ = 0.000800 (non-zero!)


In [10]:
# Interactive 3D comparison – now including the cross-gamma effect
fig = make_subplots(
  rows=2, cols=2,specs=[[{'type': 'surface'}, {'type': 'surface'}], [{'type': 'surface'}, {'type': 'surface'}]],
  subplot_titles=("True ΔV (with Cross-Gamma)", "Delta Approximation","Delta-Gamma Approximation","Approximation Error (with Cross Term)")
)

# True ΔV with cross exposure (notice the tilt!)
fig.add_trace(go.Surface(x=DS1, y=DS2, z=delta_V_true_cross, colorscale='RdBu_r', showscale=False, name='True ΔV Cross'), row=1, col=1)

# Pure Delta approximation (same as before – linear plane)
delta_approx = delta1 * DS1 + delta2 * DS2
fig.add_trace(go.Surface(x=DS1, y=DS2, z=delta_approx, colorscale='RdBu_r', showscale=False), row=1, col=2)

# Delta-Gamma approximation (now includes estimated Γ12 if you recompute it)
# For illustration we use analytical greeks + numerical cross term effect
gamma_approx_cross = (
  0.5 * gamma11 * DS1**2 +
  gamma12_num * DS1 * DS2 +      # <-- non-zero cross term
  0.5 * gamma22 * DS2**2
)

delta_gamma_approx_cross = delta_approx + gamma_approx_cross
fig.add_trace(go.Surface(x=DS1, y=DS2, z=delta_gamma_approx_cross, colorscale='RdBu_r', showscale=False), row=2, col=1)

# Error surface – much smaller thanks to quadratic term
error_cross = delta_V_true_cross - delta_gamma_approx_cross
fig.add_trace(go.Surface(x=DS1, y=DS2, z=error_cross, colorscale='RdBu_r', showscale=True), row=2, col=2)

fig.update_layout(
  title="Delta-Gamma Risk Surfaces with Non-Zero Cross-Gamma (Tilted Interaction)",
  height=950,
  scene=dict(xaxis_title='ΔS₁ (%)', yaxis_title='ΔS₂ (%)', zaxis_title='ΔV'),
  scene2=dict(xaxis_title='ΔS₁ (%)', yaxis_title='ΔS₂ (%)', zaxis_title='ΔV'),
  scene3=dict(xaxis_title='ΔS₁ (%)', yaxis_title='ΔS₂ (%)', zaxis_title='ΔV'),
  scene4=dict(xaxis_title='ΔS₁ (%)', yaxis_title='ΔS₂ (%)', zaxis_title='Error')
)
fig.show()